# Specim Hyperspectral Imaging Processing

## Enviroment Setup

In [1]:
import os
import sys
import time
import cv2
import torch
import csv
import json
import shutil
import threading
import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from typing import Optional, List, Tuple, Set
from __future__ import annotations
from dataclasses import dataclass, field

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.envi_zarr_conversion import convert_envi_to_zarr, convert_zarr_to_envi
from src.hsi2color_zarr import hsi_to_color_zarr
from src.reference_building_zarr import (
    build_white_reference_zarr,
    build_black_reference_zarr,
)
from src.flat_field_correction_zarr import flat_field_correction_zarr
from src.roi_mean_spectra_zarr import roi_mean_spectra_zarr
from sam2.gsam2_segmenter import GSAM2_Segmenter

In [2]:
# Check CUDA support 
!nvcc --version
!nvidia-smi

# Determine the device to use
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Print device information
if DEVICE.type == "cuda":
    print("Available Device: GPU")
    print(f"Device Name: {torch.cuda.get_device_name(DEVICE)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Capability (SM version): {torch.cuda.get_device_capability(DEVICE)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(DEVICE)} bytes")
    print(f"Memory Cached: {torch.cuda.memory_reserved(DEVICE)} bytes")
else:
    print("Available Device: CPU")

/bin/bash: line 1: nvcc: command not found
Wed May 20 10:08:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.64.01              Driver Version: 576.88         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5080        On  |   00000000:01:00.0  On |                  N/A |
| 30%   30C    P8             12W /  360W |    1557MiB /  16303MiB |      3%      Default |
|                                         |                        |                  N/A |
+----

## Helpers

In [3]:
_ENVI_HDR_EXT = ".hdr"
_ENVI_RAW_EXT = ".raw"


def _numeric_prefix(fname: str) -> Optional[int]:
    """Return the first three chars as integer if they are digits, else None."""
    base = os.path.basename(fname)
    if len(base) >= 3 and base[:3].isdigit():
        return int(base[:3])
    return None


def _is_white(name: str) -> bool:
    return "white" in name.lower()


def _is_black(name: str) -> bool:
    return "black" in name.lower() or "dark" in name.lower()


def _ensure_dir(p: str) -> None:
    os.makedirs(p, exist_ok=True)


def _load_image_paths_specim(image_folder_path):
    """Load hyperspectral image paths in the given folder."""
    # Define expected files with their prefixes
    expected_files = {
        "dark_ref_path": "DARKREF_",
        "white_ref_path": "WHITEREF_",
        "image_path": os.path.basename(image_folder_path),
    }

    # Initialize file paths
    file_paths = {key: None for key in expected_files}

    # Construct the capture path
    capture_path = os.path.join(image_folder_path, "capture")
    if not os.path.exists(capture_path):
        raise FileNotFoundError(f"'capture' folder not found under {image_folder_path}")

    # Search for files in the capture path
    for file in os.listdir(capture_path):
        for key, prefix in expected_files.items():
            if file.startswith(prefix) and file.endswith(".hdr"):
                file_paths[key] = os.path.join(capture_path, file)

    # Check if all required files are found
    for key, path in file_paths.items():
        if path is None:
            raise FileNotFoundError(
                f"No matching {key.replace('_path', '')} file found in {capture_path}"
            )

    # Return the image (.raw), dark reference(.raw), white reference(.raw)
    return [path for key, path in file_paths.items()]


def _write_image(path: str, img_rgb: np.ndarray, *, autoscale=False) -> None:
    """
    Save an RGB image:
      - handles float arrays (0..1 or arbitrary) → uint8
      - drops alpha channel if present
      - optional percentile autoscaling to avoid all-black images
    """
    _ensure_dir(os.path.dirname(path))

    x = np.asarray(img_rgb)

    # Ensure 3 channels (drop alpha, expand gray)
    if x.ndim == 2:
        x = np.stack([x, x, x], axis=-1)
    elif x.ndim == 3 and x.shape[-1] == 4:
        x = x[..., :3]

    # Convert to uint8
    if np.issubdtype(x.dtype, np.floating):
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        if autoscale:
            lo = np.percentile(x, 1)
            hi = np.percentile(x, 99)
            if hi > lo:
                x = (x - lo) / (hi - lo)
        # assume 0..1 after optional autoscale; clamp and scale
        x = np.clip(x, 0.0, 1.0)
        x = (x * 255.0 + 0.5).astype(np.uint8, copy=False)
    else:
        x = np.clip(x, 0, 255).astype(np.uint8, copy=False)

    # Write as PNG (OpenCV expects BGR)
    bgr = cv2.cvtColor(np.ascontiguousarray(x), cv2.COLOR_RGB2BGR)
    cv2.imwrite(path, bgr)


def _write_mask(path: str, mask_u8: np.ndarray) -> None:
    _ensure_dir(os.path.dirname(path))
    cv2.imwrite(path, mask_u8)


def _to_uint8_rgb(img):
    import numpy as np

    x = np.asarray(img)

    # Channel-first -> channel-last
    if x.ndim == 3 and x.shape[0] in (3, 4) and x.shape[-1] not in (3, 4):
        x = np.moveaxis(x, 0, -1)

    # Grayscale -> 3-channel
    if x.ndim == 2:
        x = np.stack([x, x, x], axis=-1)

    # Drop alpha if present
    if x.ndim == 3 and x.shape[-1] == 4:
        x = x[..., :3]

    # Convert dtype/range to uint8 [0,255]
    if np.issubdtype(x.dtype, np.floating):
        # If already 0..1, scale to 0..255; otherwise clip to 0..255
        x_min, x_max = float(np.nanmin(x)), float(np.nanmax(x))
        if 0.0 <= x_min and x_max <= 1.0:
            x = x * 255.0
        x = np.clip(x, 0.0, 255.0).astype(np.uint8, copy=False)
    else:
        x = np.clip(x, 0, 255).astype(np.uint8, copy=False)

    return np.ascontiguousarray(x)


def _safe_remove(p: str) -> None:
    try:
        if os.path.isdir(p):
            shutil.rmtree(p)
        elif os.path.exists(p):
            os.remove(p)
    except Exception:
        pass


def _prune_temp_workspace(temp_root: str, keep_paths: tuple[str, ...] = ()) -> None:
    """
    Delete EVERYTHING under temp_root except the paths listed in keep_paths
    (and their children). Use this at the end of each loop iteration.
    """
    if not os.path.isdir(temp_root):
        return
    keep_abs = {os.path.abspath(k) for k in keep_paths if k}
    for name in os.listdir(temp_root):
        p = os.path.abspath(os.path.join(temp_root, name))
        # preserve refs; delete everything else
        if any(p == k or p.startswith(k + os.sep) for k in keep_abs):
            continue
        _safe_remove(p)


def _save_roi_mean_std_plot(
    wavelengths, means_list, stds_list, ids=None, title=None, save_path=None, show=False
):
    import numpy as np
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    wl = np.asarray(wavelengths, dtype=float)
    if ids is None:
        ids = range(1, len(means_list) + 1)

    fig, ax = plt.subplots(figsize=(12, 8), dpi=300)
    for rid, mu, sd in zip(ids, means_list, stds_list):
        mu = np.asarray(mu, dtype=float)
        sd = np.asarray(sd, dtype=float)
        (line,) = ax.plot(wl, mu, label=f"ROI {rid}")
        ax.fill_between(
            wl, mu - sd, mu + sd, alpha=0.2, facecolor=line.get_color(), linewidth=0
        )

    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Intensity (a.u.)")
    if title:
        ax.set_title(title)
    ax.legend()
    # ax.grid(True)

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


# Plotting for 1-row white/black reference
def _plot_reference_zarr(zarr_path: str, save_path: str, title: str):
    """
    Plot mean ±1σ spectrum from a 1-row white/black reference Zarr.
    - No relative imports (works when run as a script).
    - Robust to BIL/BIP/BSQ: detects the bands axis by matching wavelengths length.
    """
    import numpy as np
    import zarr

    ref = zarr.open(zarr_path, mode="r")
    arr = ref if isinstance(ref, zarr.Array) else ref[list(ref.array_keys())[0]]
    attrs = arr.attrs.asdict() if hasattr(arr.attrs, "asdict") else dict(arr.attrs)

    # Wavelengths
    wavelengths = attrs.get("wavelength", None)
    if wavelengths is None:
        raise ValueError(f"Missing 'wavelength' in attrs for {zarr_path}")
    wl = np.asarray(wavelengths, dtype=float).ravel()
    n_bands = wl.size

    x = np.asarray(arr[...], dtype=float)
    if x.ndim != 3:
        raise ValueError(f"Expected 3D reference cube; got shape {x.shape}")

    # Find the bands axis by matching n_bands
    band_axes = [i for i, s in enumerate(x.shape) if s == n_bands]
    if not band_axes:
        # Fallback: if none match exactly, pick the axis whose size is closest to n_bands
        band_axis = int(np.argmin([abs(s - n_bands) for s in x.shape]))
    else:
        # Prefer the last matching axis (common in BIP)
        band_axis = band_axes[-1]

    # Move bands axis to last → (R, C, B)
    x_bands_last = np.moveaxis(x, band_axis, -1)

    # Flatten spatial dims → (pixels, bands)
    pixels_by_bands = x_bands_last.reshape(-1, x_bands_last.shape[-1])

    # Mean/Std over pixels
    mu = np.nanmean(pixels_by_bands, axis=0)
    sd = np.nanstd(pixels_by_bands, axis=0)

    # Guard against length mismatch (prevents: x and y must have same first dimension)
    if mu.shape[0] != n_bands:
        # If mismatch persists, align by min length
        m = min(mu.shape[0], n_bands)
        wl_aligned = wl[:m]
        mu = mu[:m]
        sd = sd[:m]
    else:
        wl_aligned = wl

    # Use existing util to save the plot
    _save_roi_mean_std_plot(
        wl_aligned,
        [mu],
        [sd],
        ids=["Ref"],
        title=title,
        save_path=save_path,
        show=False,
    )


def _save_csv(
    path: str, wavelengths, roi_means_rows, *, ids=None, image_name: str = ""
):
    """
    Wide CSV per image:
      - First column = 'image'
      - Second column = 'id' (ROI order)
      - Remaining columns = wavelengths (means only)
    """
    _ensure_dir(os.path.dirname(path))
    header = ["image", "id"] + [str(float(w)) for w in wavelengths]
    import csv

    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        if ids is None:
            ids = range(1, len(roi_means_rows) + 1)
        for rid, row in zip(ids, roi_means_rows):
            w.writerow([image_name, int(rid)] + [float(x) for x in row])


def _merge_csvs(processed_root: str, out_csv: str, key_word: str):
    """
    Merge the single CSV found in each immediate subfolder of `processed_root`.
    Expect per-file header: ['image','id', <band1_nm>, <band2_nm>, ...]
    """
    import os, csv, json
    processed_root = os.path.abspath(processed_root)
    # #region agent log
    _debug_log = "/home/jy773/workspace/Hyper-Studio/.cursor/debug-9b438d.log"
    subdirs = [n for n in sorted(os.listdir(processed_root)) if os.path.isdir(os.path.join(processed_root, n))]
    csv_per_sub = {s: [f for f in os.listdir(os.path.join(processed_root, s)) if key_word in f and f.lower().endswith(".csv")] for s in subdirs}
    try:
        with open(_debug_log, "a") as f:
            f.write(json.dumps({"sessionId": "9b438d", "location": "specim.ipynb _merge_csvs", "message": "merge_csvs_start", "data": {"processed_root": processed_root, "key_word": key_word, "subdirs": subdirs, "csv_per_sub": csv_per_sub}, "hypothesisId": "D", "timestamp": __import__("time").time() * 1000}) + "\n")
    except Exception:
        pass
    # #endregion
    rows = []
    header = None

    for name in sorted(os.listdir(processed_root)):
        sub = os.path.join(processed_root, name)
        if not os.path.isdir(sub):
            continue

        csv_files = [
            f for f in os.listdir(sub) if key_word in f and f.lower().endswith(".csv")
        ]
        if not csv_files:
            continue
        if len(csv_files) != 1:
            raise RuntimeError(
                f"Expected exactly one {key_word} CSV in {sub}, found: {csv_files}"
            )

        csv_path = os.path.join(sub, csv_files[0])
        with open(csv_path, "r", newline="") as f:
            r = csv.reader(f)
            try:
                file_header = next(r)  # read header
            except StopIteration:
                continue

            if header is None:
                header = file_header  # keep first file's header
            else:
                # skip header if same
                if file_header != header:
                    raise RuntimeError(f"Header mismatch in {csv_path}")

            for row in r:
                if row:
                    rows.append(row)

    if not rows or header is None:
        raise RuntimeError(f"No CSVs found under {processed_root}")

    os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
    with open(out_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)

In [4]:
def _compute_srgb_xyz_lab(
    zarr_path: str,
    illuminants_path: str,
    illuminant_d: int = 65,
    truncate_nm: float = 780.0,
    assume_interleave: str | None = None,
    sort_wavelengths: bool = True,
    exposure: float = 1.0,
    normalize: str = "none",
):
    """
    Returns sRGB, XYZ, Lab.
    """
    srgb = hsi_to_color_zarr(
        zarr_path,
        illuminants_path,
        illuminant_d=illuminant_d,
        assume_interleave=assume_interleave,
        truncate_nm=truncate_nm,
        sort_wavelengths=sort_wavelengths,
        normalize=normalize,
        exposure=exposure,
        output="srgb",
        compute=True,
    )
    xyz = hsi_to_color_zarr(
        zarr_path,
        illuminants_path,
        illuminant_d=illuminant_d,
        assume_interleave=assume_interleave,
        truncate_nm=truncate_nm,
        sort_wavelengths=sort_wavelengths,
        normalize=normalize,
        exposure=exposure,
        output="xyz",
        compute=True,
    )
    lab = hsi_to_color_zarr(
        zarr_path,
        illuminants_path,
        illuminant_d=illuminant_d,
        assume_interleave=assume_interleave,
        truncate_nm=truncate_nm,
        sort_wavelengths=sort_wavelengths,
        normalize=normalize,
        exposure=exposure,
        output="lab",
        compute=True,
    )
    return srgb, xyz, lab


def _save_roi_color_csv(
    save_path: str,
    *,
    image_name: str,
    roi_ids: list[int],
    masks: list[np.ndarray],
    rgb: np.ndarray,  # (H, W, 3) in [0,1]
    xyz: np.ndarray,  # (H, W, 3)
    lab: np.ndarray,  # (H, W, 3)
) -> None:
    """
    Write one row per ROI with mean color under D-illuminant:
      header = image, id, r, g, b, x, y, z, l, a, b
    """
    H, W, _ = rgb.shape
    # sanity checks
    if xyz.shape[:2] != (H, W) or lab.shape[:2] != (H, W):
        raise ValueError("rgb/xyz/lab shapes mismatch.")

    with open(save_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["image", "id", "R", "G", "B", "X", "Y", "Z", "L*", "a*", "b*"])

        for rid, m in zip(roi_ids, masks):
            mask = (m > 0) if m.dtype != bool else m
            if mask.shape != (H, W):
                raise ValueError(f"ROI mask shape {mask.shape} != image shape {(H, W)}")

            # channel means with NaN-safe handling for empty masks
            def ch_mean(arr3: np.ndarray, c: int) -> float:
                vals = arr3[..., c][mask]
                return float(np.nan) if vals.size == 0 else float(vals.mean())

            r = ch_mean(rgb, 0)
            g = ch_mean(rgb, 1)
            b = ch_mean(rgb, 2)
            x = ch_mean(xyz, 0)
            y = ch_mean(xyz, 1)
            z = ch_mean(xyz, 2)
            L = ch_mean(lab, 0)
            a = ch_mean(lab, 1)
            bb = ch_mean(lab, 2)

            writer.writerow([image_name, int(rid), r, g, b, x, y, z, L, a, bb])

## Processing Pipeline

In [5]:
@dataclass
class PipelineState:
    processed_root: str
    white_ref_zarr: Optional[str] = None
    black_ref_zarr: Optional[str] = None
    stale_white_paths: List[str] = field(default_factory=list)
    stale_black_paths: List[str] = field(default_factory=list)


@dataclass
class PipelineCfg:
    temp_root: str
    save_ffc: bool
    assume_interleave: Optional[str]
    illuminants_mat_path: str
    illuminant_d: int


@dataclass
class SegmenterConfig:
    text_prompt: str
    hf_model_id: str
    detector_device: str
    sam2_cfg_path: str
    sam2_ckpt_path: str
    sam2_device: str
    box_threshold: float
    max_dets: int
    multimask_output: bool


# -------- single-image processor --------
def process_hdr(
    hdr_path: str,
    seg: GSAM2_Segmenter,
    seg_cfg: SegmenterConfig,
    pipeline_cfg: PipelineCfg,
    state: PipelineState,
    *,
    input_folder_name_for_meta: str,
    overwrite: bool = True,
) -> Optional[str]:
    """
    Process exactly one ENVI pair (hdr+raw) pointed to by `hdr_path`.
    """
    base = os.path.splitext(os.path.basename(hdr_path))[0]  # e.g., "001-white"
    is_white = _is_white(base)
    is_black = _is_black(base)

    # Per-image output folder
    out_dir = os.path.join(state.processed_root, base)
    if os.path.exists(out_dir):
        if not overwrite:
            return None  # skip already-processed
    _ensure_dir(out_dir)

    # Convert to Zarr (temp)
    _ensure_dir(pipeline_cfg.temp_root)
    zarr_path = os.path.join(pipeline_cfg.temp_root, f"{base}.zarr")
    convert_envi_to_zarr(hdr_path, zarr_path, chunks=(256, 256, 256), overwrite=True)

    # --- WHITE reference ---
    if is_white:
        rowref_path = os.path.join(pipeline_cfg.temp_root, f"{base}_rowref_white.zarr")
        white_ret = build_white_reference_zarr(
            white_zarr_path=zarr_path,
            illuminants_path=pipeline_cfg.illuminants_mat_path,
            interleave=pipeline_cfg.assume_interleave,
            normalize="global",
            out_rowref_zarr=rowref_path,
            overwrite=True,
        )
        if state.white_ref_zarr and os.path.abspath(
            state.white_ref_zarr
        ) != os.path.abspath(rowref_path):
            state.stale_white_paths.append(state.white_ref_zarr)
        state.white_ref_zarr = rowref_path

        # Save returned RGB + mask (best-effort)
        try:
            rgb = white_ret.get("rgb", None)
            if rgb is not None:
                _write_image(os.path.join(out_dir, base + "_rgb.png"), rgb)
            mask = white_ret.get("mask", None)
            if mask is not None:
                mask_u8 = (
                    (mask.astype(np.uint8) * 255) if mask.dtype != np.uint8 else mask
                )
                _write_mask(os.path.join(out_dir, base + "_mask.png"), mask_u8)
        except Exception:
            pass

        # Plot white reference spectrum
        _plot_reference_zarr(
            zarr_path=rowref_path,
            save_path=os.path.join(out_dir, f"{base}_white_ref_plot.png"),
            title="White Reference Mean ±1σ",
        )

        # Export row-ref white as ENVI
        convert_zarr_to_envi(
            state.white_ref_zarr, os.path.join(out_dir, f"{base}_rowref.raw")
        )

        # cleanup temp cube
        try:
            os.remove(zarr_path)
        except Exception:
            pass
        return out_dir

    # --- BLACK reference ---
    if is_black:
        rowref_path = os.path.join(pipeline_cfg.temp_root, f"{base}_rowref_black.zarr")
        build_black_reference_zarr(
            zarr_path,
            rowref_path,
            interleave=pipeline_cfg.assume_interleave,
            overwrite=True,
        )
        if state.black_ref_zarr and os.path.abspath(
            state.black_ref_zarr
        ) != os.path.abspath(rowref_path):
            state.stale_black_paths.append(state.black_ref_zarr)
        state.black_ref_zarr = rowref_path

        # Plot black reference spectrum
        _plot_reference_zarr(
            zarr_path=rowref_path,
            save_path=os.path.join(out_dir, f"{base}_black_ref_plot.png"),
            title="Black Reference Mean ±1σ",
        )

        # Export row-ref black as ENVI
        convert_zarr_to_envi(
            state.black_ref_zarr, os.path.join(out_dir, f"{base}_rowref.raw")
        )

        try:
            os.remove(zarr_path)
        except Exception:
            pass
        return out_dir

    # --- SAMPLE: require refs first ---
    if not state.white_ref_zarr or not state.black_ref_zarr:
        try:
            os.remove(zarr_path)
        finally:
            raise RuntimeError(
                f"Missing reference(s) before processing sample '{base}'. "
                f"white_ref_zarr={state.white_ref_zarr}, black_ref_zarr={state.black_ref_zarr}"
            )

    # 1) FFC -> Zarr
    ffc_zarr = os.path.join(pipeline_cfg.temp_root, f"{base}_ffc.zarr")
    flat_field_correction_zarr(
        zarr_path,
        state.white_ref_zarr,
        state.black_ref_zarr,
        ffc_zarr,
        interleave=pipeline_cfg.assume_interleave,
    )
    if pipeline_cfg.save_ffc:
        convert_zarr_to_envi(ffc_zarr, os.path.join(out_dir, f"{base}_ffc.raw"))

    # 2) HSI -> RGB
    rgb, xyz, lab = _compute_srgb_xyz_lab(
        zarr_path=ffc_zarr,
        illuminants_path=pipeline_cfg.illuminants_mat_path,
        illuminant_d=pipeline_cfg.illuminant_d,
        assume_interleave=pipeline_cfg.assume_interleave,
        exposure=1.0,
        normalize="none",
    )
    _write_image(os.path.join(out_dir, base + "_rgb.png"), rgb)

    # 3) Segment to get masks
    masks, annotated_bgr, mask_images, stacked_mask = seg.segment(
        rgb=_to_uint8_rgb(rgb), text_prompt=seg_cfg.text_prompt, visualize=False
    )

    # Save annotations/masks
    np.save(os.path.join(out_dir, base + "masks.npy"), masks)
    cv2.imwrite(os.path.join(out_dir, base + "annotated.png"), annotated_bgr)
    _write_mask(os.path.join(out_dir, base + "_mask_stacked.png"), stacked_mask)

    # 4) ROI mean spectra/color
    roi_means_rows, roi_stds_rows, roi_ids = [], [], []
    wavelengths_ref = None
    for idx, m in enumerate(masks, start=1):
        roi_mask = (m > 0) if m.dtype != bool else m
        lam, mu, sd = roi_mean_spectra_zarr(
            mask=roi_mask, zarr_path=ffc_zarr, interleave=pipeline_cfg.assume_interleave
        )
        if wavelengths_ref is None:
            wavelengths_ref = lam
        roi_means_rows.append(mu)
        roi_stds_rows.append(sd)
        roi_ids.append(idx)

    _save_csv(
        os.path.join(out_dir, base + "_roi_spectra.csv"),
        wavelengths_ref,
        roi_means_rows,
        ids=roi_ids,
        image_name=f"{input_folder_name_for_meta}_{base}",
    )
    _save_roi_mean_std_plot(
        wavelengths_ref,
        roi_means_rows,
        roi_stds_rows,
        ids=roi_ids,
        title="ROI Mean ±1σ",
        save_path=os.path.join(out_dir, base + "_roi_mean_std.png"),
        show=False,
    )

    _save_roi_color_csv(
        os.path.join(out_dir, base + "_roi_color.csv"),
        image_name=f"{input_folder_name_for_meta}_{base}",
        roi_ids=roi_ids,
        masks=masks,
        rgb=rgb,
        xyz=xyz,
        lab=lab,
    )

    # 5) cleanup temp zarrs for this image (keep refs alive)
    _prune_temp_workspace(
        pipeline_cfg.temp_root, keep_paths=(state.white_ref_zarr, state.black_ref_zarr)
    )

    # 6) persist metadata
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(
            {
                "source_hdr": hdr_path,
                "order_index": _numeric_prefix(base),
                "white_ref_zarr": state.white_ref_zarr,
                "black_ref_zarr": state.black_ref_zarr,
                "prompt": seg_cfg.text_prompt,
                "hf_model_id": seg_cfg.hf_model_id,
                "sam2_cfg_path": seg_cfg.sam2_cfg_path,
                "sam2_ckpt_path": seg_cfg.sam2_ckpt_path,
                "box_threshold": seg_cfg.box_threshold,
                "max_dets": seg_cfg.max_dets,
            },
            f,
            indent=2,
        )

In [6]:
def process_folder(
    input_folder: str,
    seg: GSAM2_Segmenter,
    seg_cfg: SegmenterConfig,
    pipeline_cfg: PipelineCfg,
) -> str:
    """
    Run the batch pipeline on all ENVI pairs in `input_folder/capture`.
    """
    input_folder = os.path.abspath(input_folder)
    parent = os.path.dirname(input_folder.rstrip(os.sep))
    in_name = os.path.basename(input_folder.rstrip(os.sep))
    processed_root = os.path.join(parent, f"{in_name}_processed")
    _ensure_dir(processed_root)

    state = PipelineState(processed_root=processed_root)

    _ensure_dir(pipeline_cfg.temp_root)

    hdr_list = _load_image_paths_specim(input_folder)
    if not hdr_list:
        raise RuntimeError(f"No ENVI .hdr/.raw pairs found in {input_folder}")

    for hdr_path in tqdm(
        hdr_list, desc=f"Processing images in {input_folder}", unit="image"
    ):
        try:
            process_hdr(
                hdr_path,
                seg=seg,
                seg_cfg=seg_cfg,
                pipeline_cfg=pipeline_cfg,
                state=state,
                input_folder_name_for_meta=os.path.basename(input_folder),
            )
        except Exception as e:
            # Optional: log and continue or re-raise depending on your tolerance
            print(f"[WARN] Failed to process {hdr_path}: {e}")

    # After loop: prune temp (allow refs to be cleaned now)
    try:
        _prune_temp_workspace(pipeline_cfg.temp_root)
    except Exception as e:
        print(f"Warning: Failed to prune temp workspace: {e}")

    # #region agent log
    try:
        _debug_log = "/home/jy773/workspace/Hyper-Studio/.cursor/debug-9b438d.log"
        subdirs_pre = [n for n in sorted(os.listdir(processed_root))] if os.path.isdir(processed_root) else []
        with open(_debug_log, "a") as _f:
            _f.write(json.dumps({"sessionId": "9b438d", "location": "specim.ipynb process_folder", "message": "before_merge_csvs", "data": {"processed_root": processed_root, "in_name": in_name, "subdirs": subdirs_pre}, "hypothesisId": "D", "timestamp": __import__("time").time() * 1000}) + "\n")
    except Exception:
        pass
    # #endregion
    # Merge all CSVs
    try:
        _merge_csvs(
            processed_root,
            os.path.join(processed_root, f"{in_name}_all_roi_spectra.csv"),
            "spectra",
        )
        _merge_csvs(
            processed_root,
            os.path.join(processed_root, f"{in_name}_all_roi_color.csv"),
            "color",
        )
    except Exception as e:
        print(f"Warning: Failed to merge CSVs: {e}")

    # Remove temp folder (best-effort)
    try:
        os.rmdir(pipeline_cfg.temp_root)
    except Exception:
        pass

    return processed_root

## Main

### Pipeline Config

In [7]:
# Pipeline configuration
pipeline_cfg = PipelineCfg(
    temp_root="temp",
    save_ffc=True,
    assume_interleave=None,
    illuminants_mat_path="./src/reference/D_illuminants.mat",
    illuminant_d=65,
)

### GSAM Instance

In [ ]:
# Build GSAM2 segmenter instance
seg_cfg = SegmenterConfig(
    text_prompt="callus",
    hf_model_id="IDEA-Research/grounding-dino-base",
    detector_device=DEVICE.type,
    sam2_cfg_path="configs/sam2.1/sam2.1_hiera_l.yaml",
    sam2_ckpt_path="../resources/sam2/checkpoints/sam2.1_hiera_large.pt",
    sam2_device=DEVICE.type,
    box_threshold=0.30,
    max_dets=30,
    multimask_output=False,
)

seg = GSAM2_Segmenter(
    hf_model_id=seg_cfg.hf_model_id,
    detector_device=seg_cfg.detector_device,
    sam2_cfg_path=seg_cfg.sam2_cfg_path,
    sam2_ckpt_path=seg_cfg.sam2_ckpt_path,
    sam2_device=seg_cfg.sam2_device,
    box_threshold=seg_cfg.box_threshold,
    max_dets=seg_cfg.max_dets,
    multimask_output=seg_cfg.multimask_output,
)

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/1206 [00:00<?, ?it/s]

### Process Folders

In [9]:
# Input folder to process
base_folder = "/mnt/d/Embryogenic Callus Development"

for folder in os.listdir(base_folder):
    process_folder(
        input_folder=os.path.join(base_folder, folder),
        seg=seg,
        seg_cfg=seg_cfg,
        pipeline_cfg=pipeline_cfg,
    )
    
_merge_csvs(
            base_folder,
            os.path.join(base_folder, f"all_roi_spectra.csv"),
            "spectra",
        )

Processing images in /mnt/d/Embryogenic Callus Development/czy:   0%|          | 0/3 [00:00<?, ?image/s]

Processing images in /mnt/d/Embryogenic Callus Development/czy:  33%|███▎      | 1/3 [00:00<00:01,  1.03image/s]RuntimeWarning: invalid value encountered in divide
Processing images in /mnt/d/Embryogenic Callus Development/czy:  67%|██████▋   | 2/3 [00:02<00:01,  1.13s/image]UserWarning: Memory efficient kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:960.)
Falling back to all available kernels for scaled_dot_product_attention (which may have a slower speed).
Processing images in /mnt/d/Embryogenic Callus Development/czy: 100%|██████████| 3/3 [00:50<00:00, 16.77s/image]
Processing images in /mnt/d/Embryogenic Callus Development/czy_b:  33%|███▎      | 1/3 [00:00<00:01,  1.10image/s]RuntimeWarning: invalid value encountered in divide
Processing images in /mnt/d/Embryogenic Callus Development/czy_b: 100%|██████████| 3/3 [00:47<00:00, 15.91s/image]
Processing images in /mnt/d/Embryogenic Callus Development/rzy:  33%|███▎     

[WARN] Failed to process /mnt/d/Embryogenic Callus Development/rzy/capture/rzy.hdr: 'NoneType' object is not iterable


Processing images in /mnt/d/Embryogenic Callus Development/rzy_b:  33%|███▎      | 1/3 [00:00<00:01,  1.17image/s]RuntimeWarning: invalid value encountered in divide
Processing images in /mnt/d/Embryogenic Callus Development/rzy_b: 100%|██████████| 3/3 [00:42<00:00, 14.02s/image]


[WARN] Failed to process /mnt/d/Embryogenic Callus Development/rzy_b/capture/rzy_b.hdr: 'NoneType' object is not iterable


...

[WARN] Failed to process /mnt/d/Embryogenic Callus Development/W-G_b/capture/W-G_b.hdr: 'NoneType' object is not iterable


Processing images in /mnt/d/Embryogenic Callus Development/W-M:  33%|███▎      | 1/3 [00:00<00:01,  1.20image/s]RuntimeWarning: invalid value encountered in divide
Processing images in /mnt/d/Embryogenic Callus Development/W-M: 100%|██████████| 3/3 [00:45<00:00, 15.09s/image]


[WARN] Failed to process /mnt/d/Embryogenic Callus Development/W-M/capture/W-M.hdr: 'NoneType' object is not iterable


Processing images in /mnt/d/Embryogenic Callus Development/W-M_b:  33%|███▎      | 1/3 [00:00<00:01,  1.09image/s]RuntimeWarning: invalid value encountered in divide
Processing images in /mnt/d/Embryogenic Callus Development/W-M_b: 100%|██████████| 3/3 [00:43<00:00, 14.50s/image]

[WARN] Failed to process /mnt/d/Embryogenic Callus Development/W-M_b/capture/W-M_b.hdr: 'NoneType' object is not iterable
